# 🌫️ Spatio-Temporal Air Pollution Forecasting
## Graph Neural Networks (GCN) + LSTM | USA EPA Dataset

### Pipeline Overview
```
EPA CSV Data → PySpark Processing → HBase (HappyBase) → Graph Construction → GNN+LSTM Model → R Visualization
```

| Stage | Tool | Purpose |
|---|---|---|
| Ingestion & Cleaning | PySpark | Handle 150M+ rows |
| Storage | HBase + HappyBase | Time-series key-value store |
| Graph ML | PyTorch Geometric | Spatial dependencies |
| Temporal ML | PyTorch LSTM | Temporal patterns |
| Visualization | R (ggplot2, leaflet) | Maps & forecast plots |

---
## 📦 Stage 0 — Environment Setup & Imports

In [3]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install pyspark happybase torch torch-geometric scikit-learn pandas
#              numpy matplotlib seaborn tqdm requests
# For PyTorch Geometric:
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv \
#              torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cpu.html

import os, warnings, requests, zipfile, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as TF
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# PyTorch Geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

# Scikit-learn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import BallTree

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f"✅ PyTorch device: {DEVICE}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ NumPy version: {np.__version__}")

✅ PyTorch device: mps
✅ PyTorch version: 2.8.0
✅ NumPy version: 2.2.6


---
## 📥 Stage 1 — Data Download (USA EPA AQS)

We download the **hourly PM2.5** dataset from EPA's public FTP.
- URL pattern: `https://aqs.epa.gov/aqsweb/airdata/hourly_88101_{year}.zip`
- Parameter 88101 = PM2.5 (Local Conditions)
- Each year ~ 6-10 million rows

In [ ]:
# ── EPA Data Downloader ────────────────────────────────────────────────────────
DATA_DIR = Path("data/epa_raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

POLLUTANTS = {
    "pm25":  "88101",   # PM2.5 - Local Conditions
    "no2":   "42602",   # Nitrogen Dioxide
    "ozone": "44201",   # Ozone
    "co":    "42101",   # Carbon Monoxide
}

YEARS = [2022, 2023]   # Add more years for bigger data

def download_epa_data(param_code: str, year: int, pollutant_name: str) -> Path:
    """
    Downloads hourly EPA AQS data for a given pollutant and year.
    Returns local path to the extracted CSV.
    """
    url = f"https://aqs.epa.gov/aqsweb/airdata/hourly_{param_code}_{year}.zip"
    out_csv = DATA_DIR / f"hourly_{pollutant_name}_{year}.csv"

    if out_csv.exists():
        print(f"  ↳ Already exists: {out_csv.name}")
        return out_csv

    print(f"  ↳ Downloading {pollutant_name} {year} from EPA...")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_name = z.namelist()[0]
        z.extract(csv_name, DATA_DIR)
        extracted = DATA_DIR / csv_name
        extracted.rename(out_csv)

    print(f"  ↳ Saved → {out_csv}")
    return out_csv


downloaded_files = {}
for pollutant, code in POLLUTANTS.items():
    downloaded_files[pollutant] = []
    for year in YEARS:
        try:
            path = download_epa_data(code, year, pollutant)
            downloaded_files[pollutant].append(str(path))
        except Exception as e:
            print(f"  ✗ Failed {pollutant} {year}: {e}")

print("\n✅ Download complete.")
for p, files in downloaded_files.items():
    print(f"  {p}: {len(files)} file(s)")

  ↳ Downloading pm25 2022 from EPA...
  ↳ Saved → data/epa_raw/hourly_pm25_2022.csv
  ↳ Downloading pm25 2023 from EPA...


---
## ⚡ Stage 2 — Big Data Processing with Apache PySpark

In [ ]:
# ── Initialize Spark Session ───────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("AirPollutionForecasting")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark version: {spark.version}")
print(f"   UI: http://localhost:4040")

In [ ]:
# ── EPA Schema Definition ──────────────────────────────────────────────────────
epa_schema = StructType([
    StructField("state_code",           StringType(),  True),
    StructField("county_code",          StringType(),  True),
    StructField("site_num",             StringType(),  True),
    StructField("parameter_code",       IntegerType(), True),
    StructField("poc",                  IntegerType(), True),
    StructField("latitude",             DoubleType(),  True),
    StructField("longitude",            DoubleType(),  True),
    StructField("datum",                StringType(),  True),
    StructField("parameter_name",       StringType(),  True),
    StructField("date_local",           StringType(),  True),
    StructField("time_local",           StringType(),  True),
    StructField("date_gmt",             StringType(),  True),
    StructField("time_gmt",             StringType(),  True),
    StructField("sample_measurement",   DoubleType(),  True),
    StructField("units_of_measure",     StringType(),  True),
    StructField("mdl",                  DoubleType(),  True),
    StructField("uncertainty",          DoubleType(),  True),
    StructField("qualifier",            StringType(),  True),
    StructField("method_type",          StringType(),  True),
    StructField("method_code",          StringType(),  True),
    StructField("method_name",          StringType(),  True),
    StructField("state_name",           StringType(),  True),
    StructField("county_name",          StringType(),  True),
    StructField("date_of_last_change",  StringType(),  True),
])

print("✅ Schema defined — 24 columns")

In [ ]:
# ── Load & Clean PM2.5 Data in Spark ──────────────────────────────────────────
def load_and_clean_epa_spark(csv_paths: list, spark_session) -> "DataFrame":
    """
    Loads multiple EPA CSV files into a single Spark DataFrame.
    Applies:
      1. Timestamp parsing
      2. Station ID creation
      3. Quality filters (remove negatives, nulls)
      4. Deduplication
    """
    df = (
        spark_session.read
        .option("header", "true")
        .schema(epa_schema)
        .csv(csv_paths)
    )

    # ── 1. Create unified timestamp ────────────────────────────────────────────
    df = df.withColumn(
        "timestamp",
        F.to_timestamp(
            F.concat(F.col("date_local"), F.lit(" "), F.col("time_local")),
            "yyyy-MM-dd HH:mm"
        )
    )

    # ── 2. Unique station ID ───────────────────────────────────────────────────
    df = df.withColumn(
        "station_id",
        F.concat(
            F.col("state_code"), F.lit("-"),
            F.col("county_code"), F.lit("-"),
            F.col("site_num")
        )
    )

    # ── 3. Quality filter ──────────────────────────────────────────────────────
    df = (
        df.filter(F.col("sample_measurement").isNotNull())
          .filter(F.col("sample_measurement") >= 0)
          .filter(F.col("sample_measurement") <= 500)   # physical upper bound µg/m³
          .filter(F.col("timestamp").isNotNull())
          .filter(F.col("latitude").isNotNull())
    )

    # ── 4. Select & deduplicate ────────────────────────────────────────────────
    df = (
        df.select(
            "station_id", "timestamp",
            "latitude", "longitude",
            "state_name", "county_name",
            "sample_measurement"
        )
        .dropDuplicates(["station_id", "timestamp"])
        .orderBy("station_id", "timestamp")
    )

    return df


pm25_df = load_and_clean_epa_spark(
    downloaded_files["pm25"], spark
)
pm25_df.cache()
print(f"✅ PM2.5 rows loaded: {pm25_df.count():,}")
pm25_df.show(5, truncate=False)

In [ ]:
# ── Feature Engineering in Spark ──────────────────────────────────────────────
def engineer_features_spark(df):
    """
    Adds temporal features, rolling statistics, and lag features.
    All computed in a distributed fashion via PySpark.
    """
    # ── Temporal cyclical encoding ─────────────────────────────────────────────
    df = (
        df
        .withColumn("hour",        F.hour("timestamp"))
        .withColumn("day_of_week", F.dayofweek("timestamp"))
        .withColumn("month",       F.month("timestamp"))
        .withColumn("is_weekend",  (F.dayofweek("timestamp").isin([1, 7])).cast(IntegerType()))
        # Sine/cosine encoding for cyclical features
        .withColumn("hour_sin",   F.sin(2 * np.pi * F.col("hour") / 24))
        .withColumn("hour_cos",   F.cos(2 * np.pi * F.col("hour") / 24))
        .withColumn("month_sin",  F.sin(2 * np.pi * F.col("month") / 12))
        .withColumn("month_cos",  F.cos(2 * np.pi * F.col("month") / 12))
    )

    # ── Rolling statistics per station ────────────────────────────────────────
    station_window = (
        Window.partitionBy("station_id")
              .orderBy(F.col("timestamp").cast("long"))
              .rowsBetween(-23, 0)   # 24-hour window
    )
    df = (
        df
        .withColumn("pm25_24h_mean", F.mean("sample_measurement").over(station_window))
        .withColumn("pm25_24h_std",  F.stddev("sample_measurement").over(station_window))
        .withColumn("pm25_24h_max",  F.max("sample_measurement").over(station_window))
    )

    # ── Lag features (1h, 3h, 6h, 12h, 24h ago) ───────────────────────────────
    lag_window = (
        Window.partitionBy("station_id")
              .orderBy(F.col("timestamp").cast("long"))
    )
    for lag in [1, 3, 6, 12, 24]:
        df = df.withColumn(f"pm25_lag_{lag}h", F.lag("sample_measurement", lag).over(lag_window))

    # ── Drop rows with nulls introduced by lag ────────────────────────────────
    df = df.dropna(subset=["pm25_lag_24h"])

    return df


pm25_features = engineer_features_spark(pm25_df)
print(f"✅ Feature engineering done. Columns: {len(pm25_features.columns)}")
pm25_features.printSchema()

In [ ]:
# ── Spark Summary Statistics ───────────────────────────────────────────────────
print("📊 PM2.5 Distribution Statistics:")
pm25_features.select("sample_measurement").describe().show()

print("\n📍 Top 10 States by Station Count:")
(
    pm25_features
    .groupBy("state_name")
    .agg(F.countDistinct("station_id").alias("n_stations"),
         F.count("*").alias("n_readings"),
         F.mean("sample_measurement").alias("mean_pm25"))
    .orderBy(F.desc("n_stations"))
    .show(10)
)

In [ ]:
# ── Filter to well-covered stations & save as Parquet ─────────────────────────
# Keep stations with at least 8,000 hourly readings (~333 days coverage)
MIN_READINGS = 8000

station_counts = (
    pm25_features
    .groupBy("station_id", "latitude", "longitude", "state_name")
    .agg(F.count("*").alias("n_readings"))
    .filter(F.col("n_readings") >= MIN_READINGS)
)

pm25_filtered = pm25_features.join(
    station_counts.select("station_id"),
    on="station_id", how="inner"
)

n_stations = station_counts.count()
n_rows     = pm25_filtered.count()
print(f"✅ Stations retained: {n_stations}")
print(f"✅ Total rows after filter: {n_rows:,}")

# ── Save processed data as Parquet (columnar, compressed, fast) ───────────────
PARQUET_PATH = "data/processed/pm25_hourly.parquet"
(
    pm25_filtered
    .write
    .mode("overwrite")
    .partitionBy("state_name")
    .parquet(PARQUET_PATH)
)
print(f"✅ Saved Parquet → {PARQUET_PATH}")

# Also save station metadata as CSV for graph construction
station_meta = station_counts.toPandas()
station_meta.to_csv("data/processed/station_metadata.csv", index=False)
print(f"✅ Station metadata saved: {len(station_meta)} stations")

---
## 🗄️ Stage 3 — HBase Storage via HappyBase

HBase row key design:
```
station_id#YYYYMMDD#HH  →  e.g., "06-037-1103#20230115#08"
```
Column families:
- `cf:pm25`         — raw measurement
- `cf:features`     — engineered features
- `cf:meta`         — lat/lon, state

In [ ]:
# ── HappyBase HBase Client ─────────────────────────────────────────────────────
# NOTE: Requires HBase running locally or via Docker:
#   docker run -d -p 2181:2181 -p 16000:16000 -p 16020:16020 \
#              -p 9090:9090 harisekhon/hbase
# Then pip install happybase

import happybase

HBASE_HOST = 'localhost'
TABLE_NAME  = 'air_quality'

def setup_hbase_table(host: str, table_name: str):
    """Creates HBase table with column families if it doesn't exist."""
    connection = happybase.Connection(host)
    existing   = [t.decode() for t in connection.tables()]

    if table_name not in existing:
        connection.create_table(
            table_name,
            {
                'cf_pm25':     dict(max_versions=3),
                'cf_features': dict(max_versions=1),
                'cf_meta':     dict(max_versions=1),
            }
        )
        print(f"✅ HBase table '{table_name}' created.")
    else:
        print(f"ℹ️  HBase table '{table_name}' already exists.")

    connection.close()


def write_to_hbase(pandas_df: pd.DataFrame, host: str, table_name: str,
                   batch_size: int = 1000):
    """
    Batch-writes a pandas DataFrame into HBase.
    Row key: station_id#YYYYMMDD#HH
    """
    connection = happybase.Connection(host)
    table      = connection.table(table_name)

    written = 0
    with table.batch(batch_size=batch_size) as batch:
        for _, row in tqdm(pandas_df.iterrows(), total=len(pandas_df),
                           desc="Writing to HBase"):
            ts   = pd.Timestamp(row["timestamp"])
            rkey = f"{row['station_id']}#{ts.strftime('%Y%m%d')}#{ts.strftime('%H')}"

            batch.put(
                rkey.encode(),
                {
                    # Column family: pm25
                    b'cf_pm25:value':         str(row['sample_measurement']).encode(),

                    # Column family: features
                    b'cf_features:hour_sin':  str(row.get('hour_sin', '')).encode(),
                    b'cf_features:hour_cos':  str(row.get('hour_cos', '')).encode(),
                    b'cf_features:month_sin': str(row.get('month_sin', '')).encode(),
                    b'cf_features:month_cos': str(row.get('month_cos', '')).encode(),
                    b'cf_features:lag_1h':    str(row.get('pm25_lag_1h', '')).encode(),
                    b'cf_features:lag_24h':   str(row.get('pm25_lag_24h', '')).encode(),
                    b'cf_features:roll_mean': str(row.get('pm25_24h_mean', '')).encode(),

                    # Column family: meta
                    b'cf_meta:latitude':      str(row['latitude']).encode(),
                    b'cf_meta:longitude':     str(row['longitude']).encode(),
                    b'cf_meta:state':         str(row['state_name']).encode(),
                }
            )
            written += 1

    connection.close()
    print(f"✅ Written {written:,} rows to HBase table '{table_name}'")


def read_station_from_hbase(station_id: str, date_from: str,
                            date_to: str, host: str, table_name: str) -> pd.DataFrame:
    """
    Reads time-series for a station over a date range using row key scan.
    date_from / date_to format: 'YYYYMMDD'
    """
    connection = happybase.Connection(host)
    table      = connection.table(table_name)

    start_key = f"{station_id}#{date_from}#00".encode()
    stop_key  = f"{station_id}#{date_to}#23".encode()

    records = []
    for key, data in table.scan(row_start=start_key, row_stop=stop_key):
        records.append({
            'row_key':    key.decode(),
            'pm25':       float(data.get(b'cf_pm25:value', b'nan')),
            'lag_1h':     float(data.get(b'cf_features:lag_1h', b'nan')),
            'roll_mean':  float(data.get(b'cf_features:roll_mean', b'nan')),
            'latitude':   float(data.get(b'cf_meta:latitude', b'0')),
            'longitude':  float(data.get(b'cf_meta:longitude', b'0')),
        })

    connection.close()
    return pd.DataFrame(records)


# ── Demo write (first 5000 rows to HBase) ─────────────────────────────────────
# NOTE: Comment out if HBase is not running — we'll fall back to Parquet for modeling
try:
    sample_pdf = pm25_filtered.limit(5000).toPandas()
    setup_hbase_table(HBASE_HOST, TABLE_NAME)
    write_to_hbase(sample_pdf, HBASE_HOST, TABLE_NAME)
except Exception as e:
    print(f"⚠️  HBase not available (start Docker container): {e}")
    print("   Continuing with Parquet-based pipeline...")

---
## 🕸️ Stage 4 — Graph Construction

- **Nodes**: Each monitoring station
- **Edges**: Stations within `RADIUS_KM` kilometers (spatial adjacency)
- **Edge weights**: Inverse distance (closer = stronger connection)
- **Node features**: Last `T=12` hours of PM2.5 + temporal encodings

In [ ]:
# ── Load processed data for modeling ──────────────────────────────────────────
# Read Parquet back into pandas for graph construction + PyTorch
try:
    pm25_pd = spark.read.parquet(PARQUET_PATH).toPandas()
except:
    pm25_pd = pm25_filtered.toPandas()   # fallback if parquet write is pending

pm25_pd["timestamp"] = pd.to_datetime(pm25_pd["timestamp"])
station_meta = pd.read_csv("data/processed/station_metadata.csv")

print(f"✅ Data loaded: {pm25_pd.shape}")
print(f"   Stations: {pm25_pd['station_id'].nunique()}")
print(f"   Date range: {pm25_pd['timestamp'].min()} → {pm25_pd['timestamp'].max()}")

In [ ]:
# ── Pivot: [timestamp × station] matrix ───────────────────────────────────────
pivot = (
    pm25_pd
    .pivot_table(index="timestamp", columns="station_id",
                 values="sample_measurement", aggfunc="mean")
)

# Forward-fill then backward-fill gaps (short outages)
pivot = pivot.ffill(limit=6).bfill(limit=6)

# Drop stations still missing >20% of timesteps
thresh  = int(0.80 * len(pivot))
pivot   = pivot.dropna(axis=1, thresh=thresh)
pivot   = pivot.fillna(pivot.median())   # remaining with median

stations = pivot.columns.tolist()
N = len(stations)
print(f"✅ Pivot matrix: {pivot.shape}  ({N} stations × {len(pivot)} timesteps)")

In [ ]:
# ── Build Spatial Graph ────────────────────────────────────────────────────────
RADIUS_KM = 150.0   # Connect stations within 150 km

# Coordinates for each station in the pivot
station_loc = (
    pm25_pd
    .drop_duplicates("station_id")
    .set_index("station_id")
    [["latitude", "longitude"]]
    .reindex(stations)    # align with pivot columns
)

coords_rad = np.radians(station_loc[["latitude", "longitude"]].values)

# BallTree for fast haversine nearest-neighbor search
tree = BallTree(coords_rad, metric="haversine")
RADIUS_RAD = RADIUS_KM / 6371.0   # Earth radius in km

edge_index_list = []
edge_weight_list = []

for i, coord in enumerate(coords_rad):
    indices, distances = tree.query_radius(
        coord.reshape(1, -1), r=RADIUS_RAD, return_distance=True
    )
    for j, dist in zip(indices[0], distances[0]):
        if i != j:   # no self-loops
            dist_km = dist * 6371.0
            weight  = 1.0 / (dist_km + 1e-5)   # inverse distance
            edge_index_list.append([i, j])
            edge_weight_list.append(weight)

edge_index  = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)

# Normalize edge weights
edge_weight = edge_weight / edge_weight.max()

print(f"✅ Graph built:")
print(f"   Nodes (stations): {N}")
print(f"   Edges: {edge_index.shape[1]:,}")
print(f"   Avg degree: {edge_index.shape[1] / N:.1f} neighbors/station")

In [ ]:
# ── Visualize graph connectivity ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Degree distribution
degrees  = np.bincount(edge_index[0].numpy())
axes[0].hist(degrees, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel("Node Degree (# neighbors)", fontsize=12)
axes[0].set_ylabel("Count", fontsize=12)
axes[0].set_title("Station Degree Distribution", fontsize=14, fontweight='bold')
axes[0].axvline(degrees.mean(), color='red', ls='--', label=f'Mean={degrees.mean():.1f}')
axes[0].legend()

# Station map with edges (sample 500 edges for clarity)
lats = station_loc['latitude'].values
lons = station_loc['longitude'].values
axes[1].scatter(lons, lats, s=10, c='steelblue', alpha=0.7, zorder=5)

sample_edges = edge_index_list[:500]
for (i, j) in sample_edges:
    axes[1].plot([lons[i], lons[j]], [lats[i], lats[j]],
                 'gray', alpha=0.1, linewidth=0.5)

axes[1].set_xlabel("Longitude", fontsize=12)
axes[1].set_ylabel("Latitude", fontsize=12)
axes[1].set_title(f"Station Graph (radius={RADIUS_KM}km)", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig("data/graph_structure.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graph visualization saved.")

---
## 🧩 Stage 5 — Dataset Preparation for GNN+LSTM

In [ ]:
# ── Normalize PM2.5 values ─────────────────────────────────────────────────────
T_IN  = 12   # Input: past 12 hours
T_OUT = 1    # Output: predict next 1 hour (can be 6 or 24)

scaler = MinMaxScaler(feature_range=(0, 1))
values = pivot.values   # shape: [timesteps, n_stations]
values_scaled = scaler.fit_transform(values)   # normalize across all stations

print(f"✅ Value matrix shape: {values_scaled.shape}")
print(f"   Min: {values_scaled.min():.3f} | Max: {values_scaled.max():.3f}")

In [ ]:
# ── Sliding Window Dataset ─────────────────────────────────────────────────────
class SpatioTemporalDataset(Dataset):
    """
    Creates sliding window samples from the [timestep × station] matrix.
    Each sample:
      X: [T_IN, N_stations]  — past T_IN hours for ALL stations
      y: [N_stations]        — next-hour PM2.5 for ALL stations
    """
    def __init__(self, data: np.ndarray, t_in: int, t_out: int):
        self.data  = torch.tensor(data, dtype=torch.float32)
        self.t_in  = t_in
        self.t_out = t_out
        self.len   = len(data) - t_in - t_out + 1

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.t_in]                       # [T_IN, N]
        y = self.data[idx + self.t_in : idx + self.t_in + self.t_out]  # [T_OUT, N]
        y = y.squeeze(0)                                            # [N] for T_OUT=1
        return x, y


# ── Train / Val / Test split (70 / 15 / 15) ────────────────────────────────────
T_total = len(values_scaled)
t_train = int(0.70 * T_total)
t_val   = int(0.85 * T_total)

train_data = values_scaled[:t_train]
val_data   = values_scaled[t_train:t_val]
test_data  = values_scaled[t_val:]

train_ds = SpatioTemporalDataset(train_data, T_IN, T_OUT)
val_ds   = SpatioTemporalDataset(val_data,   T_IN, T_OUT)
test_ds  = SpatioTemporalDataset(test_data,  T_IN, T_OUT)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)

print(f"✅ Dataset splits:")
print(f"   Train: {len(train_ds):,} samples")
print(f"   Val:   {len(val_ds):,} samples")
print(f"   Test:  {len(test_ds):,} samples")

---
## 🧠 Stage 6 — GNN + LSTM Model Definition

In [ ]:
# ── Spatio-Temporal GNN-LSTM Architecture ─────────────────────────────────────
class GCNLayer(nn.Module):
    """Single Graph Convolutional layer with residual connection."""
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.gcn     = GCNConv(in_dim, out_dim)
        self.bn      = nn.BatchNorm1d(out_dim)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x, edge_index, edge_weight=None):
        h = self.gcn(x, edge_index, edge_weight)
        h = self.bn(h)
        h = TF.relu(h)
        h = self.dropout(h)
        return h + self.residual(x)


class STGNN_LSTM(nn.Module):
    """
    Spatio-Temporal Graph Neural Network with LSTM.

    Architecture:
      For each time step t in [0, T_IN):
        1. GCN layers: capture spatial dependencies across stations
        2. Collect GCN outputs across all time steps
      3. LSTM: model temporal dynamics of the spatial embeddings
      4. FC head: predict PM2.5 at t+1 for all stations

    Input:  X  [batch, T_IN, N_stations]
    Output: ŷ  [batch, N_stations]
    """
    def __init__(
        self,
        n_stations:    int,
        t_in:          int,
        gcn_hidden:    int  = 64,
        gcn_layers:    int  = 2,
        lstm_hidden:   int  = 128,
        lstm_layers:   int  = 2,
        dropout:       float = 0.2,
    ):
        super().__init__()
        self.n_stations  = n_stations
        self.t_in        = t_in
        self.gcn_hidden  = gcn_hidden

        # GCN stack (applied per time step, weights shared across time)
        gcn_dims = [1] + [gcn_hidden] * gcn_layers
        self.gcn_layers = nn.ModuleList([
            GCNLayer(gcn_dims[i], gcn_dims[i+1], dropout)
            for i in range(gcn_layers)
        ])

        # LSTM: input is the GCN embedding for all nodes flattened
        self.lstm = nn.LSTM(
            input_size  = n_stations * gcn_hidden,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            batch_first = True,
            dropout     = dropout if lstm_layers > 1 else 0.0,
        )

        # Output head
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden // 2, n_stations),
            nn.Sigmoid()   # output in [0,1] to match MinMaxScaler
        )

    def forward(self, x, edge_index, edge_weight=None):
        """
        x:           [batch, T_IN, N]
        edge_index:  [2, E]
        edge_weight: [E]
        Returns:     [batch, N]
        """
        B, T, N = x.shape
        gcn_outs = []

        for t in range(T):
            x_t  = x[:, t, :].reshape(B * N, 1)     # [B*N, 1]
            # Expand edge_index for batched graphs
            ei_batch = self._expand_edge_index(edge_index, B, N)
            ew_batch = edge_weight.repeat(B) if edge_weight is not None else None

            h = x_t
            for gcn in self.gcn_layers:
                h = gcn(h, ei_batch, ew_batch)       # [B*N, gcn_hidden]

            h = h.reshape(B, N * self.gcn_hidden)    # [B, N*gcn_hidden]
            gcn_outs.append(h)

        # Stack time steps → [B, T, N*gcn_hidden]
        gcn_seq = torch.stack(gcn_outs, dim=1)

        # LSTM over time
        lstm_out, _ = self.lstm(gcn_seq)             # [B, T, lstm_hidden]
        last_hidden  = lstm_out[:, -1, :]            # [B, lstm_hidden]

        return self.head(last_hidden)                # [B, N]

    @staticmethod
    def _expand_edge_index(edge_index, batch_size, n_nodes):
        """Repeat edge_index for each graph in the batch."""
        ei_list = []
        for b in range(batch_size):
            ei_list.append(edge_index + b * n_nodes)
        return torch.cat(ei_list, dim=1)


# ── Instantiate model ──────────────────────────────────────────────────────────
model = STGNN_LSTM(
    n_stations  = N,
    t_in        = T_IN,
    gcn_hidden  = 64,
    gcn_layers  = 2,
    lstm_hidden = 128,
    lstm_layers = 2,
    dropout     = 0.2,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model: STGNN-LSTM")
print(f"   Parameters: {total_params:,}")
print(model)

---
## 🏋️ Stage 7 — Training Loop

In [ ]:
# ── Training Configuration ─────────────────────────────────────────────────────
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 8    # early stopping patience

optimizer  = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                patience=4, verbose=True)
criterion  = nn.HuberLoss(delta=0.5)   # robust to outliers

# Move graph to device
edge_index_dev  = edge_index.to(DEVICE)
edge_weight_dev = edge_weight.to(DEVICE)


def run_epoch(loader, model, optimizer, criterion, edge_index,
              edge_weight, device, train=True):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)    # [B, T_IN, N]
            y_batch = y_batch.to(device)    # [B, N]

            if train:
                optimizer.zero_grad()

            pred = model(x_batch, edge_index, edge_weight)   # [B, N]
            loss = criterion(pred, y_batch)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches


# ── Training Loop ──────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": []}
best_val_loss = float('inf')
no_improve    = 0

print(f"🚀 Training for up to {EPOCHS} epochs on {DEVICE}...\n")

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(
        train_loader, model, optimizer, criterion,
        edge_index_dev, edge_weight_dev, DEVICE, train=True
    )
    val_loss = run_epoch(
        val_loader, model, optimizer, criterion,
        edge_index_dev, edge_weight_dev, DEVICE, train=False
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    scheduler.step(val_loss)

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve    = 0
        torch.save(model.state_dict(), "data/best_stgnn_lstm.pt")
        flag = " ← best"
    else:
        no_improve += 1
        flag = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{EPOCHS} | "
              f"Train: {train_loss:.5f} | Val: {val_loss:.5f}{flag}")

    # ── Early stopping ────────────────────────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print(f"\n✅ Best validation loss: {best_val_loss:.5f}")

In [ ]:
# ── Plot Training Curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train Loss', color='steelblue', linewidth=2)
ax.plot(history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Huber Loss", fontsize=12)
ax.set_title("STGNN-LSTM Training Curves", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("data/training_curves.png", dpi=150)
plt.show()

---
## 📊 Stage 8 — Evaluation & Metrics

In [ ]:
# ── Load best model & run test evaluation ─────────────────────────────────────
model.load_state_dict(torch.load("data/best_stgnn_lstm.pt", map_location=DEVICE))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(DEVICE)
        pred = model(x_batch, edge_index_dev, edge_weight_dev).cpu().numpy()
        all_preds.append(pred)
        all_targets.append(y_batch.numpy())

preds   = np.concatenate(all_preds,   axis=0)   # [test_samples, N]
targets = np.concatenate(all_targets, axis=0)   # [test_samples, N]

# ── Inverse transform to original µg/m³ scale ─────────────────────────────────
preds_inv   = scaler.inverse_transform(preds)
targets_inv = scaler.inverse_transform(targets)

# ── Metrics ────────────────────────────────────────────────────────────────────
mae  = mean_absolute_error(targets_inv.flatten(), preds_inv.flatten())
rmse = np.sqrt(mean_squared_error(targets_inv.flatten(), preds_inv.flatten()))
r2   = r2_score(targets_inv.flatten(), preds_inv.flatten())
mape = np.mean(np.abs((targets_inv - preds_inv) / (targets_inv + 1e-5))) * 100

print("═" * 45)
print("  📈  Test Set Evaluation — STGNN-LSTM")
print("═" * 45)
print(f"  MAE  : {mae:.3f} µg/m³")
print(f"  RMSE : {rmse:.3f} µg/m³")
print(f"  MAPE : {mape:.2f}%")
print(f"  R²   : {r2:.4f}")
print("═" * 45)

In [ ]:
# ── Forecast vs Actual — Sample Station ───────────────────────────────────────
SAMPLE_STATION_IDX = 0   # change to any station index
PLOT_HOURS         = 168  # 1 week

y_true = targets_inv[:PLOT_HOURS, SAMPLE_STATION_IDX]
y_pred = preds_inv[:PLOT_HOURS,   SAMPLE_STATION_IDX]
hours  = np.arange(PLOT_HOURS)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time series comparison
axes[0].plot(hours, y_true, label='Actual PM2.5',    color='steelblue', linewidth=1.5)
axes[0].plot(hours, y_pred, label='Predicted PM2.5', color='tomato',    linewidth=1.5, ls='--')
axes[0].fill_between(hours, y_true, y_pred, alpha=0.2, color='orange')
axes[0].set_xlabel("Hour", fontsize=11)
axes[0].set_ylabel("PM2.5 (µg/m³)", fontsize=11)
axes[0].set_title(f"1-Hour Ahead PM2.5 Forecast — Station: {stations[SAMPLE_STATION_IDX]}",
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].axhline(35, color='red', ls=':', alpha=0.5, label='EPA 24h Standard')

# Scatter plot
axes[1].scatter(y_true, y_pred, alpha=0.4, s=15, color='steelblue')
lim = max(y_true.max(), y_pred.max()) * 1.05
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=2, label='Perfect forecast')
axes[1].set_xlabel("Actual PM2.5 (µg/m³)", fontsize=11)
axes[1].set_ylabel("Predicted PM2.5 (µg/m³)", fontsize=11)
axes[1].set_title("Predicted vs Actual Scatter", fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].text(0.05, 0.92, f"R² = {r2:.3f}\nMAE = {mae:.2f} µg/m³",
             transform=axes[1].transAxes, fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.tight_layout()
plt.savefig("data/forecast_vs_actual.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-Station MAE Analysis ───────────────────────────────────────────────────
station_maes = mean_absolute_error(
    targets_inv, preds_inv, multioutput='raw_values'
)

station_eval = pd.DataFrame({
    'station_id': stations,
    'mae':        station_maes,
})
station_eval = station_eval.merge(station_meta[["station_id","latitude","longitude","state_name"]],
                                   on="station_id", how="left")

fig, ax = plt.subplots(figsize=(14, 5))
sc = ax.scatter(
    station_eval['longitude'], station_eval['latitude'],
    c=station_eval['mae'], cmap='RdYlGn_r',
    s=50, alpha=0.8, edgecolors='white', linewidths=0.3
)
plt.colorbar(sc, ax=ax, label='MAE (µg/m³)')
ax.set_title("Per-Station Forecast MAE — USA", fontsize=14, fontweight='bold')
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("data/per_station_mae_map.png", dpi=150)
plt.show()

print("\nTop 10 hardest stations to forecast:")
print(station_eval.nlargest(10, 'mae')[["station_id","state_name","mae"]].to_string(index=False))

In [ ]:
# ── Baseline Comparison: Persistence Model ────────────────────────────────────
# Persistence: predict t+1 = t (simplest possible baseline)
persist_preds   = targets_inv[:-1]   # shift by 1
persist_targets = targets_inv[1:]

persist_mae  = mean_absolute_error(persist_targets.flatten(), persist_preds.flatten())
persist_rmse = np.sqrt(mean_squared_error(persist_targets.flatten(), persist_preds.flatten()))

model_maes  = [mae,  persist_mae]
model_rmses = [rmse, persist_rmse]
labels      = ["STGNN-LSTM (Ours)", "Persistence Baseline"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
colors = ['steelblue', 'lightcoral']

for i, (ax, metric, vals, unit) in enumerate([
    (axes[0], "MAE",  model_maes,  "µg/m³"),
    (axes[1], "RMSE", model_rmses, "µg/m³"),
]):
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f"{metric} Comparison", fontsize=13, fontweight='bold')
    ax.set_ylabel(f"{metric} ({unit})", fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha='center', va='bottom', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Model vs Baseline", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("data/baseline_comparison.png", dpi=150)
plt.show()

print(f"\n📉 MAE improvement over persistence: "
      f"{((persist_mae - mae)/persist_mae)*100:.1f}%")

---
## 📊 Stage 9 — R Visualization Code

Save evaluation results for R, then run R scripts for publication-quality plots.

In [ ]:
# ── Export data for R visualization ───────────────────────────────────────────
SAMPLE = min(5000, len(preds_inv))

# 1. Forecast vs actual (all stations, sampled)
forecast_df = pd.DataFrame({
    'actual':    targets_inv[:SAMPLE, SAMPLE_STATION_IDX],
    'predicted': preds_inv[:SAMPLE,   SAMPLE_STATION_IDX],
    'hour':      np.arange(SAMPLE),
})
forecast_df.to_csv("data/r_forecast.csv", index=False)

# 2. Per-station evaluation metrics
station_eval.to_csv("data/r_station_metrics.csv", index=False)

# 3. Training history
pd.DataFrame(history).to_csv("data/r_training_history.csv", index=False)

print("✅ Data exported for R:")
print("   data/r_forecast.csv")
print("   data/r_station_metrics.csv")
print("   data/r_training_history.csv")

In [ ]:
# ── R Script: Full Visualization Suite ────────────────────────────────────────
r_script = '''
# ============================================================
# Air Pollution Forecasting — R Visualization Suite
# Install: install.packages(c("ggplot2","leaflet","dplyr",
#                             "viridis","plotly","htmlwidgets"))
# ============================================================

library(ggplot2)
library(dplyr)
library(leaflet)
library(plotly)
library(viridis)
library(htmlwidgets)

# ── 1. Load data ─────────────────────────────────────────────
forecast  <- read.csv("data/r_forecast.csv")
stations  <- read.csv("data/r_station_metrics.csv")
train_hist <- read.csv("data/r_training_history.csv")

# ── 2. Forecast vs Actual — ggplot2 ──────────────────────────
p1 <- ggplot(forecast[1:336,], aes(x = hour)) +
  geom_ribbon(aes(ymin = pmin(actual, predicted),
                  ymax = pmax(actual, predicted)),
              fill = "orange", alpha = 0.3) +
  geom_line(aes(y = actual,    color = "Actual"),    size = 1.0) +
  geom_line(aes(y = predicted, color = "Predicted"), size = 1.0, linetype = "dashed") +
  geom_hline(yintercept = 35, color = "red", linetype = "dotted", size = 0.8) +
  annotate("text", x = 10, y = 37, label = "EPA 24h Standard",
           color = "red", size = 3.5) +
  scale_color_manual(values = c("Actual" = "#2980B9", "Predicted" = "#E74C3C")) +
  labs(title    = "1-Hour Ahead PM2.5 Forecast — STGNN-LSTM",
       subtitle = "Two-week window, single monitoring station",
       x = "Hour", y = expression(PM[2.5] ~ (mu*g/m^3)),
       color = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top",
        plot.title       = element_text(face = "bold"),
        panel.grid.minor = element_blank())

ggsave("data/r_forecast_plot.png", p1, width = 14, height = 5, dpi = 200)

# ── 3. Training Loss Curves ───────────────────────────────────
train_long <- train_hist %>%
  mutate(epoch = row_number()) %>%
  tidyr::pivot_longer(c(train_loss, val_loss), names_to = "split", values_to = "loss")

p2 <- ggplot(train_long, aes(x = epoch, y = loss, color = split)) +
  geom_line(size = 1.2) +
  geom_point(size = 1, alpha = 0.5) +
  scale_color_manual(values = c(train_loss = "#3498DB", val_loss = "#E74C3C"),
                     labels  = c("Train", "Validation")) +
  labs(title = "STGNN-LSTM Training Curves",
       x = "Epoch", y = "Huber Loss", color = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top",
        plot.title       = element_text(face = "bold"))

ggsave("data/r_training_curves.png", p2, width = 10, height = 5, dpi = 200)

# ── 4. Scatter Plot: Predicted vs Actual ─────────────────────
p3 <- ggplot(forecast, aes(x = actual, y = predicted)) +
  geom_hex(bins = 60, aes(fill = after_stat(count))) +
  scale_fill_viridis(option = "plasma", name = "Count") +
  geom_abline(slope = 1, intercept = 0, color = "red", linetype = "dashed", size = 1) +
  labs(title    = "Predicted vs Actual PM2.5 — Density Hex Plot",
       x = expression(Actual ~ PM[2.5] ~ (mu*g/m^3)),
       y = expression(Predicted ~ PM[2.5] ~ (mu*g/m^3))) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

ggsave("data/r_scatter_hex.png", p3, width = 8, height = 7, dpi = 200)

# ── 5. Station MAE Distribution ───────────────────────────────
p4 <- ggplot(stations, aes(x = mae)) +
  geom_histogram(bins = 40, fill = "steelblue", color = "white", alpha = 0.85) +
  geom_vline(aes(xintercept = mean(mae)), color = "red",
             linetype = "dashed", size = 1) +
  annotate("text", x = mean(stations$mae) + 0.3,
           y = Inf, vjust = 2,
           label = sprintf("Mean MAE = %.2f", mean(stations$mae)),
           color = "red", size = 4) +
  labs(title = "Distribution of Per-Station MAE",
       x = expression(MAE ~ (mu*g/m^3)), y = "Number of Stations") +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

ggsave("data/r_mae_distribution.png", p4, width = 9, height = 5, dpi = 200)

# ── 6. Interactive Leaflet Map ────────────────────────────────
stations_clean <- stations %>%
  filter(!is.na(latitude), !is.na(longitude), !is.na(mae))

pal <- colorNumeric(palette = "RdYlGn",
                    domain   = stations_clean$mae,
                    reverse  = TRUE)

leaflet_map <- leaflet(stations_clean) %>%
  addProviderTiles(providers$CartoDB.Positron) %>%
  addCircleMarkers(
    lng    = ~longitude,
    lat    = ~latitude,
    radius = 6,
    color  = ~pal(mae),
    stroke = TRUE, weight = 1, opacity = 1,
    fillOpacity = 0.85,
    popup = ~paste0(
      "<b>Station:</b> ", station_id, "<br>",
      "<b>State:</b> ",   state_name,  "<br>",
      "<b>MAE:</b> ",     round(mae, 3), " µg/m³"
    ),
    label = ~station_id
  ) %>%
  addLegend(
    position = "bottomright",
    pal    = pal, values = ~mae,
    title  = "Forecast MAE (µg/m³)",
    opacity = 0.9
  ) %>%
  addControl(
    html = "<b>STGNN-LSTM: Per-Station PM2.5 Forecast Error</b>",
    position = "topright"
  )

saveWidget(leaflet_map, "data/r_station_map.html", selfcontained = TRUE)
cat("✅ R plots saved!\n")
'''

with open("data/visualize_results.R", "w") as f:
    f.write(r_script)

print("✅ R script saved → data/visualize_results.R")
print("   Run with: Rscript data/visualize_results.R")
print("   Or open in RStudio and run interactively.")

---
## 🔮 Stage 10 — Multi-Step Forecasting (6h & 24h Ahead)

In [ ]:
# ── Recursive Multi-Step Forecasting ──────────────────────────────────────────
def forecast_recursive(model, initial_window: np.ndarray,
                        edge_index, edge_weight,
                        n_steps: int, device) -> np.ndarray:
    """
    Recursively predicts n_steps ahead by feeding
    each prediction back as input.

    initial_window: [T_IN, N] — seed window (scaled)
    Returns: [n_steps, N] predicted values (scaled)
    """
    model.eval()
    window    = initial_window.copy()   # [T_IN, N]
    forecasts = []

    with torch.no_grad():
        for _ in range(n_steps):
            x = torch.tensor(window, dtype=torch.float32)\
                      .unsqueeze(0).to(device)     # [1, T_IN, N]
            y_hat = model(x, edge_index, edge_weight)  # [1, N]
            pred  = y_hat.squeeze(0).cpu().numpy()      # [N]
            forecasts.append(pred)
            # Slide window: drop oldest, append new prediction
            window = np.vstack([window[1:], pred])      # [T_IN, N]

    return np.array(forecasts)   # [n_steps, N]


# Seed from last T_IN steps of test set
seed_window = values_scaled[-T_IN:]

fc_6h  = forecast_recursive(model, seed_window, edge_index_dev,
                              edge_weight_dev, 6,  DEVICE)
fc_24h = forecast_recursive(model, seed_window, edge_index_dev,
                              edge_weight_dev, 24, DEVICE)

fc_6h_inv  = scaler.inverse_transform(fc_6h)
fc_24h_inv = scaler.inverse_transform(fc_24h)

print(f"✅ 6-hour forecast shape:  {fc_6h_inv.shape}")
print(f"✅ 24-hour forecast shape: {fc_24h_inv.shape}")

In [ ]:
# ── Plot 24-hour national average forecast ─────────────────────────────────────
nat_avg_24h = fc_24h_inv.mean(axis=1)   # average across all stations

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(nat_avg_24h, marker='o', color='steelblue', linewidth=2, markersize=6)
ax.fill_between(range(24), nat_avg_24h, alpha=0.2, color='steelblue')
ax.axhline(35, color='red', ls='--', alpha=0.7, label='EPA Standard (35 µg/m³)')
ax.set_xlabel("Hours Ahead", fontsize=12)
ax.set_ylabel("PM2.5 (µg/m³)", fontsize=12)
ax.set_title("24-Hour National Average PM2.5 Forecast", fontsize=14, fontweight='bold')
ax.set_xticks(range(24))
ax.set_xticklabels([f"+{i+1}h" for i in range(24)], rotation=45)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("data/24h_national_forecast.png", dpi=150)
plt.show()

---
## 💾 Stage 11 — Save Everything

In [ ]:
# ── Save final artifacts ───────────────────────────────────────────────────────
import pickle

# Model
torch.save({
    'model_state':    model.state_dict(),
    'model_config':   dict(n_stations=N, t_in=T_IN,
                           gcn_hidden=64, lstm_hidden=128),
    'stations':       stations,
    'edge_index':     edge_index,
    'edge_weight':    edge_weight,
    'metrics':        dict(mae=mae, rmse=rmse, r2=r2, mape=mape),
    'scaler':         scaler,
}, "data/stgnn_lstm_checkpoint.pt")

# Station metadata with results
station_eval.to_csv("data/station_evaluation.csv", index=False)

# 24h forecasts
pd.DataFrame(
    fc_24h_inv,
    columns=[f"station_{i}" for i in range(N)]
).to_csv("data/24h_forecast.csv", index=False)

print("\n" + "═" * 50)
print("  ✅  ALL ARTIFACTS SAVED")
print("═" * 50)
print("  data/stgnn_lstm_checkpoint.pt  — full model checkpoint")
print("  data/station_evaluation.csv    — per-station MAE")
print("  data/24h_forecast.csv          — 24h forecast values")
print("  data/r_*.csv                   — R visualization inputs")
print("  data/visualize_results.R       — run in RStudio")
print("═" * 50)

# Stop Spark
spark.stop()
print("\n✅ Spark session stopped.")

---
## 📋 Project Summary

| Component | Details |
|---|---|
| **Dataset** | USA EPA AQS Hourly PM2.5 (parameter 88101) |
| **Scale** | ~10M+ rows/year, 1,000+ stations |
| **Spark** | Ingestion, cleaning, lag features, rolling stats, Parquet output |
| **HBase** | Time-series store, row key = `station#date#hour` |
| **Graph** | Stations as nodes, 150km radius edges, inverse-distance weights |
| **Model** | GCN (spatial) + LSTM (temporal), shared weights per time step |
| **Metrics** | MAE, RMSE, MAPE, R² on held-out test set |
| **R Viz** | ggplot2 forecasts, leaflet station map, plotly hex density |
| **Forecast** | 1h, 6h, 24h ahead via recursive prediction |

### Key Design Decisions
- **Why GCN?** Air pollutants disperse spatially — a station's reading depends on neighbors (wind carry, industrial zones)
- **Why LSTM?** PM2.5 has strong diurnal (daily) and weekly patterns; LSTM captures this temporal memory
- **Why Huber Loss?** More robust to outlier pollution events (wildfires, industrial accidents) than MSE
- **Why MinMaxScaler?** PM2.5 values span very different ranges across stations; normalizing ensures stable training
- **Why Parquet?** Columnar format — 5-10× faster for analytical queries than CSV at this scale